# fiber-mosaic quickstart

Build named fiber photometry recordings, read fluorescence with the fiber-native API, and bundle several of them into a single experiment handle.

Layering used by the library:

- **fiber** &rarr; one channel of a recording
- **recording** &rarr; one `BaseFiberPhotometryRecording`, labelled by its `name` (each recording has its own timebase)
- **experiment** &rarr; one `FiberPhotometryRecordingGroup` holding all recordings

In [ ]:
# plotting needs the widgets extra: pip install -e .[widgets]


import numpy as np
from spikeinterface.core.numpyextractors import NumpyRecordingSegment

from fiber_mosaic import (
    BaseFiberPhotometryRecording,
    FiberPhotometryRecordingGroup,
)

## 1. Synthesize a single recording

Three fibers, 5 seconds at 100 Hz. In a real experiment the array would come from your acquisition file via a subclass of `BaseFiberPhotometryFileExtractor`.

In [ ]:
sampling_rate = 100.0  # Hz
n_samples = 500
fiber_ids = ["fiber_1", "fiber_2", "fiber_3"]

t = np.arange(n_samples) / sampling_rate
rng = np.random.default_rng(0)

green_traces = np.column_stack(
    [
        1.0 + 0.20 * np.sin(2 * np.pi * 0.5 * t),
        1.0 + 0.30 * np.sin(2 * np.pi * 0.8 * t + 1.0),
        1.0 + 0.10 * np.sin(2 * np.pi * 1.2 * t + 2.0),
    ]
).astype("float32")
green_traces += 0.02 * rng.standard_normal(green_traces.shape).astype(
    "float32"
)
green_traces.shape

## 2. Wrap the array in a segment, attach to a recording

The recording owns metadata (fiber IDs, name, sampling rate). The segment owns the actual numpy array. We reuse spikeinterface's `NumpyRecordingSegment` directly &mdash; `add_segment` wires them together, and you can attach more than one segment for multi-block experiments.

In [ ]:
green_rec = BaseFiberPhotometryRecording(
    sampling_frequency=sampling_rate,
    fiber_ids=fiber_ids,
    name="green",
    dtype="float32",
)

green_seg = NumpyRecordingSegment(
    traces=green_traces,
    sampling_frequency=sampling_rate,
    t_start=None,
)
green_rec.add_segment(green_seg)

green_rec

## 3. Read fluorescence

`get_fluorescence` is the fiber-photometry-native API. It accepts `segment_index`, `start_frame`, `end_frame`, and `fiber_ids` &mdash; no ephys-only flags like `return_in_uV`. Internally it delegates to spikeinterface's `get_traces`, so saving, slicing, and concatenation still work.

In [ ]:
# all fibers, full segment -> (n_samples, n_fibers)
all_traces = green_rec.get_fluorescence()
all_traces.shape

In [ ]:
# subset of fibers by ID
fiber_3_only = green_rec.get_fluorescence(fiber_ids=["fiber_3"])
fiber_3_only.shape

In [ ]:
# first second of data, all fibers
first_second = green_rec.get_fluorescence(start_frame=0, end_frame=100)
first_second.shape

## 4. Plot

The arrays from `get_fluorescence` plot with plain matplotlib. Section 11 shows `plot_fluorescence`, which does this for any recording, raw or processed.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 3))
for i, fid in enumerate(green_rec.fiber_ids):
    ax.plot(t, all_traces[:, i], label=fid)
ax.set_xlabel("time (s)")
ax.set_ylabel("fluorescence (a.u.)")
ax.set_title(
    f"{green_rec.name} channel \u2014 {green_rec.get_num_fibers()} fibers"
)
ax.legend(loc="upper right", fontsize=8)
fig.tight_layout()

## 5. Multi-color experiments

A real fiber photometry session usually records several colors from the same fibers &mdash; e.g. a calcium indicator (green) plus an isosbestic control (415 nm). Each color is its own recording with its own timebase. `FiberPhotometryRecordingGroup` bundles them and validates that they share the same fibers in the same order.

In [ ]:
iso_traces = (
    1.0
    + 0.05 * np.sin(2 * np.pi * 0.3 * t)[:, None]
    + 0.02 * rng.standard_normal((n_samples, 3))
).astype("float32")

iso_rec = BaseFiberPhotometryRecording(
    sampling_frequency=sampling_rate,
    fiber_ids=fiber_ids,
    name="iso",
    dtype="float32",
)
iso_rec.add_segment(
    NumpyRecordingSegment(
        traces=iso_traces,
        sampling_frequency=sampling_rate,
        t_start=None,
    )
)

experiment = FiberPhotometryRecordingGroup(
    {green_rec.name: green_rec, iso_rec.name: iso_rec}
)
experiment

## 6. Group access patterns

In [ ]:
experiment.names, experiment.fiber_ids, experiment.get_num_fibers()

In [ ]:
# index by name, then call the per-recording API
experiment["green"].get_fluorescence(fiber_ids=["fiber_3"]).shape

In [ ]:
# iterate over recordings
for name, rec in experiment.items():
    print(f"{name:>6}: {rec.get_fluorescence().shape}")

## 7. Mismatched fibers fail loudly

The group enforces that every recording shares the same fibers in the same order &mdash; downstream operations like dF/F or motion regression silently rely on this.

In [ ]:
bad = BaseFiberPhotometryRecording(
    sampling_frequency=sampling_rate,
    fiber_ids=["fiber_1", "fiber_2"],  # missing NAc
    name="red",
    dtype="float32",
)
bad.add_segment(
    NumpyRecordingSegment(
        traces=np.zeros((n_samples, 2), dtype="float32"),
        sampling_frequency=sampling_rate,
        t_start=None,
    )
)

try:
    FiberPhotometryRecordingGroup({"green": green_rec, "red": bad})
except ValueError as exc:
    print("rejected:", exc)

## 8. Timestamps with `set_times`

Timestamps follow spikeinterface: one 1-D time vector per segment, shared by all fibers. Without one, `get_times` synthesizes times from the sampling frequency and `t_start`. Acquisition clocks drift, so readers attach the file's own timestamps with `set_times`.

In [ ]:
# no explicit times yet: synthesized from the sampling frequency
print("has time vector?", green_rec.has_time_vector())
green_rec.get_times()[:3]

In [ ]:
# attach a measured clock (here: nominal times plus a little jitter)
measured = t + 1e-4 * rng.standard_normal(n_samples)
green_rec.set_times(measured, with_warning=False)

print("has time vector?", green_rec.has_time_vector())
print("start/end:", green_rec.get_start_time(), green_rec.get_end_time())
green_rec.get_times()[:3]

## 9. Discovering streams with `get_streams`

Raw photometry files (TDT, Doric, Neurophotometrics) usually bundle several **streams** &mdash; different excitation/LED colors, demodulated vs. raw signals, analog inputs, TTLs. `get_streams` is the discovery step: call it on a reader *before* constructing anything to see what is available, then hand the chosen `stream_id` / `stream_name` to that reader. The abstract base has no file format, so concrete file-reading subclasses implement it.

In [ ]:
# the abstract base reads no file format, so get_streams is not implemented
try:
    BaseFiberPhotometryRecording.get_streams("session.doric")
except NotImplementedError as exc:
    print("base is abstract:", exc)

In [ ]:
# a concrete file reader overrides it to report what is in the file
class DemoPhotometryReader(BaseFiberPhotometryRecording):
    """Demo file reader for fiber photometry data."""

    @classmethod
    def get_streams(cls, file_path):
        """Return available streams in the file.

        Parameters
        ----------
        file_path : str
            Path to the data file.

        Returns
        -------
        stream_names : list of str
            Human-readable stream names.
        stream_ids : list of str
            Stream identifiers.
        """
        # a real reader would open file_path and inspect its header here
        return ["green", "red", "iso"], ["0", "1", "2"]


names, ids = DemoPhotometryReader.get_streams("session.doric")
print("available streams:", list(zip(names, ids, strict=False)))
print(f"-> load stream id {ids[0]!r} ({names[0]!r}) with the reader")

# 10. Processing

In [ ]:
import fiber_mosaic.processing as fmproc

In [ ]:
# single recording
green_bp = fmproc.bandpass_filter(green_rec)
green_bp

In [ ]:
# single recording
green_bp_zscore = fmproc.zscore(green_bp)
green_bp_zscore

In [ ]:
# group
exp_bp = fmproc.bandpass_filter(experiment)
exp_bp

## Pipeline

A pipeline is an ordered list of named steps and their parameters, in plain JSON types, so an analysis can save exactly what it ran. For `FiberPhotometryRecordingGroup`, you can use a dict of lists with keys matching the group names.

Each entry needs to be a dictionary with the step as `name` and the params in the `params` field. Optionally, the output of each step can be tagged with `output_name`, used when cross-referencing objects from different recordings in a group.

In [ ]:
# available processing steps:
fmproc.available_processing_steps()

In [ ]:
fmproc.scale?

In [ ]:
pipeline = [
    {"name": "bandpass_filter", "params": {"freq_min": 0.2}},
    {"name": "zscore"},
    {"name": "scale", "params": {"gain": 300}}
]

In [ ]:
processed_rec = fmproc.apply_processing_pipeline(green_rec, pipeline)
processed_rec

When applied to a group, the list-based pipeline is applied to each entry:

In [ ]:
experiment

In [ ]:
processed_group = fmproc.apply_processing_pipeline(experiment, pipeline)
processed_group

In [ ]:
processed_group["iso"]

For `FiberPhotometryGroup` objects, we can use a dictionary and reference objects from other groups with the `pipeline[output_name]` notation, where `output_name`.

In this "complex" example, we have three recordings (`green` + `red` + `iso`) and we use the `iso` filtered signal for isosbestic correction of the `red`, which in turn is used to correct the `green` recording.

**This is only for demonstration!**

In [ ]:
red_traces = (
    3.0
    + 0.05 * np.sin(2 * np.pi * 0.5 * t)[:, None]
    + 0.02 * rng.standard_normal((n_samples, 3))
).astype("float32")

red_rec = BaseFiberPhotometryExtractor(
    sampling_frequency=sampling_rate,
    fiber_ids=fiber_ids,
    name="red",
    dtype="float32",
)
red_rec.add_segment(
    NumpyRecordingSegment(
        traces=red_traces,
        sampling_frequency=sampling_rate,
        t_start=None,
    )
)

experiment_complex = FiberPhotometryRecordingGroup(
    {green_rec.name: green_rec, iso_rec.name: iso_rec, red_rec.name: red_rec}
)
experiment_complex

In [ ]:
pipeline_dict_complex = {
    "green": [
        {"name": "bandpass_filter", "params": {"freq_min": 0.2}},
        {"name": "zscore"},
        {
            "name": "isosbestic_correction",
            "params": {"reference_recording": "pipeline[red_corrected]"}
        },
    ],
    "red": [
        {"name": "bandpass_filter", "params": {"freq_min": 0.2}},
        {
            "name": "isosbestic_correction",
            "params": {"reference_recording": "pipeline[iso_bandpass]"},
            "output_name": "red_corrected"
        },
    ],
    "iso": [
        {
            "name": "bandpass_filter",
            "params": {"freq_min": 0.2},
            "output_name": "iso_bandpass"
        }
    ]
}

In [ ]:
processed_group_complex = fmproc.apply_processing_pipeline(
    experiment_complex,
    pipeline_dict_complex
)
processed_group_complex

In [ ]:
print(processed_group_complex["green"].name)
print(processed_group_complex["green"]._kwargs["reference_recording"].name)
print(processed_group_complex["green"]._kwargs["reference_recording"]._kwargs["reference_recording"].name)

## 11. Plotting with `plot_fluorescence`

`plot_fluorescence` plots one or more fibers of any recording: an extractor's raw output or the result of a processing step. It wraps SpikeInterface's `plot_traces` with fiber vocabulary (`fiber_ids`, not `channel_ids`) and photometry defaults:

- the first **60 s** are shown (SpikeInterface shows 1 s), clipped to the recording;
- each fiber is **median-centered for display**, so traces on a large baseline don't flatten into a line. The data are untouched; pass `center_traces=False` to draw absolute levels.

In [ ]:
from fiber_mosaic.widgets import plot_fluorescence

plot_fluorescence(green_rec)

In [ ]:
# a subset of fibers and a time window, in seconds
plot_fluorescence(
    green_rec, fiber_ids=["fiber_1", "fiber_3"], time_range=(1, 4)
)

### Comparing processing stages

A dict draws several recordings on one axis, named by their keys. They must share fibers and timestamps, which processing steps of one recording do.

In [ ]:
plot_fluorescence(
    {
        "raw": green_rec,
        "bandpass": green_bp,
        "bandpass + zscore": green_bp_zscore,
    },
    fiber_ids=["fiber_2"],
)

### Plotting every color in a session

A `FiberPhotometryRecordingGroup` draws one subplot per color. Each color is drawn on its own timestamps, so colors sampled at different times stay aligned. As with any SpikeInterface widget, the result holds the figure and axes; `traces_widgets` holds one per color, in order.

In [ ]:
widget = plot_fluorescence(experiment)
[tw.ax.get_title() for tw in widget.traces_widgets]

In [ ]:
# overlay=True puts every color on one axis, fibers sharing a row
plot_fluorescence(fmproc.zscore(experiment), overlay=True)

### Interactive scrolling

In a notebook, `backend="ipywidgets"` adds a time slider and fiber selector. It needs the ipympl matplotlib backend, so run `%matplotlib widget` first. A group gets one interactive plot per color; `overlay=True` needs the matplotlib backend.

```python
%matplotlib widget
plot_fluorescence(green_rec, backend="ipywidgets")
```